In [3]:
from utils import * 
import subprocess
import json
from Bio.Align import PairwiseAligner 
import itertools
import requests

%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Genome analysis: **bz_0**

It is evident that sequence-based similarity clustering was insufficient to identify genes which are conserved across Betazoids. Structure-based clustering is also ineffective, as (1) many of the Betazoid proteins lack good structures, and (2) many of the important proteins (e.g. **cluster_20**) are small and primarily alpha-helical. 

We observe that conserved genes are highly-syntenous, which is also true for the genomes of filamentous phages (*Inovirideae*). Inoviruses, as well as other types of phage, tend to organize their genomes organized into modules [[Hay et. al. 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC6549030/)]. Specifically, these modules are [[Krupovic et. al. 2011](https://journals.asm.org/doi/full/10.1128/mmbr.00011-11)]:
1. **Replication module:** This module consists of pII, pX, and pV.
2. **Structure module:** This module consists of genes involved in the phage virion structure, including pVII, pIX, pVIII, pIII, and pVI.
3. **Virion morphogenesis module:** This module consists of genes involved in phage assembly and extrusion, pI (the assembly ATPase) and pIV.

In order to evaluate if a similar genomic structure is present in the Betazoids, without relying on homology-based gene clustering, we will focus analysis on a single representative Betazoid.


In [4]:
genome_id = 'bz_0'

In [5]:
genes_df = pd.read_csv('../data/genes/genes.csv', index_col=0)
genes_df['cluster_id'] = genes_df.index.map(json.load(open('../data/genes/clusters.json', 'r')))
genes_df = genes_df[genes_df.genome_id == genome_id].copy()



## InterProScan annotations

In [6]:
annot_df = pd.read_csv('../data/genes/interproscan/genes.tsv', sep='\t', header=None, names=INTERPROSCAN_FIELDS)
annot_df = annot_df[annot_df.gene_id.isin(genes_df.index)].copy()
annot_df

,gene_id,checksum,length,analysis,accession,description,start,stop,e_value,status,date,interpro_accession,interpro_description,go_terms,pathways
43,orfm.bz_0.1_82,118c3e990af003df65fa597065fe44d6,311,MobiDBLite,mobidb-lite,consensus disorder prediction,1,27,-,T,16-06-2026,-,-,-,-
49,orfm.bz_0.1_114,fa6d1af5a349acaafd2ce8f59228e36d,97,ProSiteProfiles,PS51257,Prokaryotic membrane lipoprotein lipid attachm...,1,30,6.0,T,16-06-2026,-,-,-,-
112,orfm.bz_0.1_162,8e4e1a584010bb75d40a10f6667a275d,102,SUPERFAMILY,SSF47473,EF-hand,2,61,7.74E-5,T,16-06-2026,IPR011992,EF-hand domain pair,-,-
120,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,SUPERFAMILY,SSF53098,Ribonuclease H-like,14,193,2.28E-20,T,16-06-2026,IPR012337,Ribonuclease H-like superfamily,-,-
121,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,Gene3D,G3DSA:3.30.420.10,-,9,194,1.1E-16,T,16-06-2026,IPR036397,Ribonuclease H superfamily,-,-
122,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,Pfam,PF03175,"DNA polymerase type B, organellar and viral",155,471,8.8E-38,T,16-06-2026,IPR004868,"DNA-directed DNA polymerase, family B, mitocho...",-,-
123,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,Gene3D,G3DSA:3.90.1600.10,Palm domain of DNA polymerase,371,544,6.3E-10,T,16-06-2026,IPR023211,"DNA polymerase, palm domain superfamily",-,-
124,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,Gene3D,G3DSA:3.90.1600.10,Palm domain of DNA polymerase,202,289,4.2E-10,T,16-06-2026,IPR023211,"DNA polymerase, palm domain superfamily",-,-
125,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,PANTHER,PTHR33568,DNA POLYMERASE,47,609,1.3E-37,T,16-06-2026,-,-,-,-
126,orfm.bz_0.1_98,c238be6dfbccc7577e24ec97ada0f91f,617,SUPERFAMILY,SSF56672,DNA/RNA polymerases,201,568,2.1E-52,T,16-06-2026,IPR043502,DNA/RNA polymerase superfamily,-,-


## Foldseek search 

Structures generated using the custom MSAs were used to query PDB, UniProt, SwissProt, and BFVD. 

First, we evaluated Foldseek hits obtained using the preliminary ColabFold-generated structures, which did not use any custom MSAs. We also obtained structures for a set of genes which were identified in at least 2 seperate Betazoid genomes, for which we were able to construct custom MSAs. 

In [21]:
min_plddt = 70
min_length = 50 

def get_window_mean_plddts(plddts, window_size:int=min_length, step_size:int=3):
    '''Get the mean pLDDTs of overlapping windows. This can be either per-atom (as returned by AlphaFold3) or 
    per-residue (as returned by ColabFold).
    
    :param plddts: The per-residue or per-atom pLDDTs of the stucture model. 
    :param window_size: The number of pLDDTs to pool. 
    :returns: The list of pLDDTs pooled by window. 
    '''
    windows = [plddts[i:i + window_size] for i in range(0, len(plddts), step_size)] # Split the pLDDTs into overlapping windows.
    return [np.mean(window) for window in windows]


In [23]:
# First, do another check of the structure confidences of the colabfold structures. 
colabfold_metadata_df = pd.read_csv('0-structures-colabfold_metadata.csv')
colabfold_metadata_df = colabfold_metadata_df[colabfold_metadata_df.name.isin(genes_df.index)].copy()
colabfold_metadata_df['plddts_best_model'] = colabfold_metadata_df.plddts_best_model.apply(ast.literal_eval)

colabfold_metadata_df['mean_plddt_best_model'] = colabfold_metadata_df.plddts_best_model.apply(lambda plddts : max(get_window_mean_plddts(plddts)))
colabfold_metadata_df = colabfold_metadata_df[colabfold_metadata_df.mean_plddt_best_model > min_plddt].copy()
colabfold_metadata_df = colabfold_metadata_df[colabfold_metadata_df.plddts_best_model.apply(len) > min_length].copy()

print('Number of genes with a structure suitable for Foldseek:', len(colabfold_metadata_df))

Number of genes with a structure suitable for Foldseek: 29


In [30]:
foldseek_search_df = pd.concat([pd.read_csv(path, sep='\t', names=FOLDSEEK_FIELDS).assign(path=path) for path in glob.glob('../data/genes/foldseek/genes-alphafold*')])
foldseek_search_df = foldseek_search_df[foldseek_search_df.query_id.isin(colabfold_metadata_df.name.values)].copy()

filters = dict()
filters['low_tm_score'] = foldseek_search_df.alignment_tm_score < 0.7
filters['low_query_coverage'] = foldseek_search_df.query_coverage < 0.5
filters['short_alignment'] = foldseek_search_df.alignment_length < min_length

foldseek_search_df = apply_filters(filters, foldseek_search_df)


apply_filters: 11727 entries removed by low_tm_score.
apply_filters: 1289 entries removed by low_query_coverage.
apply_filters: 284 entries removed by short_alignment.


In [ ]:
foldseek_search_df.query_id.value_counts()
foldseek_search_df.target_header.tolist()

target_header_map = dict()
target_header_map['5-methylcytosine-specific restriction endonuclease McrA'] = 'McrA'
target_header_map['Probable DNA polymerase'] = 'DNA polymerase'
target_header_map['HNH family endonuclease'] = 'HNH endonuclease'
target_header_map['Homing endonuclease'] = 'HNH endonuclease'
target_header_map['Primer-independent DNA polymerase PolB'] = 'Type B DNA polymerase'
target_header_map['CarboxypepD_reg-like domain-containing protein'] = 'Carboxypeptidase-like protein'

def clean_target_header(target_header:str):
    target_header = re.sub('AF.*model_v6', '', target_header)
    target_header = target_header.replace('(Fragment)', '')
    target_header = target_header.strip()
    target_header = re.sub('AF.*model_v6', '', target_header)
    target_header = target_header_map.get(target_header, target_header)

    target_header = 'Uncharacterized protein' if (re.search('(U|u)ncharacterized protein', target_header) is not None) else target_header

    return target_header 

foldseek_search_df['target_header'] = foldseek_search_df.target_header.apply(clean_target_header).tolist()

foldseek_search_summary_df = foldseek_search_df.groupby(['query_id', 'target_header']).apply('size').reset_index()
foldseek_search_summary_df.iloc[:50]# .target_header.iloc[0]

,query_id,target_header,0
0,orfm.bz_0.1_101,Endonuclease,2
1,orfm.bz_0.1_101,HNH domain-containing protein,22
2,orfm.bz_0.1_101,HNH endonuclease,40
3,orfm.bz_0.1_101,HNH nuclease domain-containing protein,21
4,orfm.bz_0.1_101,McrA,1
5,orfm.bz_0.1_101,Nuclease associated modular domain-containing ...,1
6,orfm.bz_0.1_101,Putative HNH endonuclease,3
7,orfm.bz_0.1_101,Uncharacterized protein,4
8,orfm.bz_0.1_132,Carboxypeptidase-like protein,2
9,orfm.bz_0.1_132,Carboxypeptidase-like regulatory domain-contai...,6


In [ ]:
target_header_map = dict()

target_header_map['uncharacterized'] = foldseek_search_df.target_header.str.contains('uncharacterized', case=False)
target_header_map['AAA family ATPase'] = foldseek_search_df.target_header.str.contains('AAA family ATPase|AAA-domain-containing|ATP-binding|AAA')
target_header_map['ATPase'] = foldseek_search_df.target_header.str.contains('ATPase') & ~target_header_map['AAA family ATPase']
target_header_map['transposase'] = foldseek_search_df.target_header.str.contains('transposase', case=False)
target_header_map['filamentous phage protein'] = foldseek_search_df.target_header.str.contains('Cf1c|Spiroplasmavirus|Plectrovirus|assembly', case=False)

target_header_map = list(zip(*list(target_header_map.items())))
foldseek_search_df['category'] = np.select(condlist=target_header_map[-1], choicelist=target_header_map[0], default='none')

# foldseek_search_df[foldseek_search_df.query_end < 200].sort_values('alignment_tm_score').drop_duplicates('target_id').iloc[-100:][['query_length', 'target_length', 'query_start', 'query_end', 'target_end', 'target_header']]